# Plan e.B -- Model B: ARDL(p,q) Bounds Testing

Specifies and estimates the ARDL(p,q) bounds-testing model (Pesaran, Shin & Smith, 2001) that
becomes the project's **primary model for inference** whenever stationarity testing (step iii)
finds a mix of I(0)/I(1) regressors with no I(2) variable (Branch B). ARDL bounds testing
estimates short-run dynamics and a long-run cointegrating relationship in one step and remains
valid regardless of whether individual regressors are I(0) or I(1) -- the project's substantive
answer to the spurious-regression risk in Model A.

See `docs/2_plan/modeling/b_model_b_ardl_bounds_testing.md` for the full spec. This notebook is
a standalone, literal execution of that spec -- independent of `vi_estimation.ipynb`, which also
estimates Model B as part of a larger execution checklist alongside Model A. Both notebooks
implement the identical Model B formula and write to the same output files; if they ever
disagree, the plan doc (not either notebook) is authoritative on what the model is.

**Input** (`outputs/`):
- `analysis_frame_1990_2024.csv` (from step i -- the 1990-2024 analysis frame, N=35)
- `modeling_path_decision.csv` (from step v -- confirms Branch B was taken, and which
  regressors are I(1))
- `model_a_static_ols_coefficients.csv` / `model_a_static_ols_fit_stats.csv` (from
  `a_model_a_static_ols.ipynb` -- needed for step 9's Model A vs. Model B contrast)

**Outputs** (`outputs/`):
- `ardl_lag_selection.csv` -- AIC/BIC by candidate `(p,q)`
- `ardl_bounds_test.csv` -- bounds F-test at the AIC-selected `(p,q)`
- `ardl_long_run_coefficients.csv` -- long-run coefficients, delta-method SEs, at the
  AIC-selected `(p,q)`
- `ardl_error_correction_term.csv` -- `theta1` (ECT) at the AIC-selected `(p,q)`
- `ardl_short_run_coefficients.csv` -- short-run coefficients at the AIC-selected `(p,q)`
- `ardl_capped_1_1_bounds_test.csv` / `..._long_run_coefficients.csv` / `..._error_correction_term.csv`
  / `..._short_run_coefficients.csv` -- the same five outputs at a fixed, leanest ARDL(1,1)
  specification (see the degrees-of-freedom note before Step 8)
- `model_b_first_differenced_ols_coefficients.csv` / `..._fit_stats.csv` -- the required
  comparison OLS on first-differenced variables


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tools import add_constant
from statsmodels.tsa.ardl import ARDL, UECM

ROOT = Path.cwd().resolve().parents[1]
OUTPUT_DIR = ROOT / "outputs"

FRAME_IN = OUTPUT_DIR / "analysis_frame_1990_2024.csv"
DECISION_IN = OUTPUT_DIR / "modeling_path_decision.csv"
MODEL_A_COEF_IN = OUTPUT_DIR / "model_a_static_ols_coefficients.csv"
MODEL_A_FIT_IN = OUTPUT_DIR / "model_a_static_ols_fit_stats.csv"

LAG_SELECTION_OUT = OUTPUT_DIR / "ardl_lag_selection.csv"
BOUNDS_TEST_OUT = OUTPUT_DIR / "ardl_bounds_test.csv"
LONG_RUN_OUT = OUTPUT_DIR / "ardl_long_run_coefficients.csv"
ECT_OUT = OUTPUT_DIR / "ardl_error_correction_term.csv"
SHORT_RUN_OUT = OUTPUT_DIR / "ardl_short_run_coefficients.csv"
CAPPED_BOUNDS_TEST_OUT = OUTPUT_DIR / "ardl_capped_1_1_bounds_test.csv"
CAPPED_LONG_RUN_OUT = OUTPUT_DIR / "ardl_capped_1_1_long_run_coefficients.csv"
CAPPED_ECT_OUT = OUTPUT_DIR / "ardl_capped_1_1_error_correction_term.csv"
CAPPED_SHORT_RUN_OUT = OUTPUT_DIR / "ardl_capped_1_1_short_run_coefficients.csv"
DIFF_OLS_COEF_OUT = OUTPUT_DIR / "model_b_first_differenced_ols_coefficients.csv"
DIFF_OLS_FIT_OUT = OUTPUT_DIR / "model_b_first_differenced_ols_fit_stats.csv"

SIGNIFICANCE_LEVELS = [(0.01, "***"), (0.05, "**"), (0.10, "*")]


def stars(p_value: float) -> str:
    for threshold, mark in SIGNIFICANCE_LEVELS:
        if p_value < threshold:
            return mark
    return ""


# Same six regressors and same dependent variable as Model A -- only the functional form
# (differenced + one-period lagged levels) differs. See a_model_a_static_ols.md's spec table
# for variable roles/hypotheses/expected signs.
REGRESSORS = {
    "DIVP": "divp",
    "DIVM": "divm",
    "INF": "inflation_rate_pct",
    "EXR": "exchange_rate",
    "log(FDI)": "log_fdi",
    "SHOCK": "shock",
}


## Step 1 -- Confirm trigger (Branch B) before proceeding

In [2]:
frame = pd.read_csv(FRAME_IN).set_index("year")
assert frame.shape[0] == 35, f"expected 35-row analysis frame, got {frame.shape[0]}"

eri = frame["eri"].rename("ERI")
X = frame[list(REGRESSORS.values())].rename(columns={v: k for k, v in REGRESSORS.items()})

decision = pd.read_csv(DECISION_IN).iloc[0]
branch = decision["branch"]
i1_vars = [v.strip() for v in str(decision["i1_variables"]).split(",") if v.strip()]

print(f"Branch recorded in step v: {branch}")
print(f"I(1) regressors (drive the spurious-regression risk in Model A and are the ones whose "
      f"long-run coefficients are meaningful here): {i1_vars}")

if branch == "C":
    raise RuntimeError(
        "Branch C was selected in step v -- ARDL bounds testing is invalid with an I(2) "
        "regressor. This notebook stops here per the plan's trigger condition; Model B is not "
        "estimated at all for a Branch C run."
    )
if branch == "A":
    print(
        "\nNOTE: Branch A (all six variables I(0)) was recorded -- per the plan's trigger "
        "condition, Model B is not estimated for a Branch A run either (static OLS on levels "
        "is the sufficient single model). The cells below still execute for completeness/"
        "cross-check purposes, but their output is not applicable to the Chapter 4 draft under "
        "Branch A."
    )
else:
    assert branch == "B", f"unexpected branch value: {branch!r}"
    print("\nBranch B confirmed -- proceeding with ARDL(p,q) bounds testing as the primary model.")


Branch recorded in step v: B
I(1) regressors (drive the spurious-regression risk in Model A and are the ones whose long-run coefficients are meaningful here): ['DIVP', 'DIVM', 'EXR', 'log(FDI)']

Branch B confirmed -- proceeding with ARDL(p,q) bounds testing as the primary model.


## Model specification

Conditional Error-Correction (ECM) form of the ARDL(p,q) model:

```
ΔERI_t = α0 + Σγᵢ·ΔERI_(t-i) + Σδⱼ·ΔX_(t-j) + θ1·ERI_(t-1) + θ2·DIVP_(t-1) + θ3·DIVM_(t-1)
         + θ4·INF_(t-1) + θ5·EXR_(t-1) + θ6·log(FDI)_(t-1) + θ7·SHOCK_(t-1) + ε_t
```

where `X = {DIVP, DIVM, INF, EXR, log(FDI), SHOCK}` -- the same six regressors and dependent
variable as Model A; only the functional form differs. `θ1` is the error-correction term
(speed of adjustment); `θ2...θ7` are used to derive the long-run coefficients
`βₖ_LR = -θₖ/θ1`.

Estimated via `statsmodels.tsa.ardl.UECM` -- the unrestricted-ECM reparametrization of
ARDL(p,q) with the built-in Pesaran/Shin/Smith bounds test, the direct equivalent of EViews'
ARDL wizard in "conditional ECM" output form, which is what every coefficient here is
cross-checked against.


## Step 2 -- Lag selection

Grid search over `p` (own lags of `ERI`) and `q` (lags of each regressor, applied uniformly
across all six rather than optimized independently per regressor -- with only ~34 observations,
letting each regressor pick its own lag independently would explode the search space and risks
overfitting the lag structure to noise). Max lag capped at 1-2 per the plan, given the ~34-35
observation sample -- protecting degrees of freedom is the binding constraint here, not model
fit. `q = 0` is excluded: `UECM` requires at least one lag for every exogenous regressor to
construct the conditional-ECM short-run terms.

AIC and BIC are compared on a common sample (`hold_back` fixed at the largest candidate `p`, so
all candidates are fit on the same 33 observations -- otherwise AIC/BIC are not comparable
across different lag lengths).


In [3]:
MAX_P = 2
lag_grid = []
for p in (1, 2):
    for q in (1, 2):
        fit = ARDL(eri, lags=p, exog=X, order=q, trend="c", hold_back=MAX_P).fit()
        lag_grid.append({
            "p": p, "q": q,
            "n_obs": int(fit.nobs),
            "k_params": len(fit.params),
            "df_resid": int(fit.df_resid),
            "aic": fit.aic,
            "bic": fit.bic,
        })

lag_selection = pd.DataFrame(lag_grid)
lag_selection.to_csv(LAG_SELECTION_OUT, index=False)
print(f"Written -> {LAG_SELECTION_OUT}")
lag_selection


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_lag_selection.csv


,p,q,n_obs,k_params,df_resid,aic,bic
0,1,1,33,14,19,-32.305969,-9.858356
1,1,2,33,20,13,-40.295026,-8.868367
2,2,1,33,15,18,-35.152112,-11.207991
3,2,2,33,21,12,-39.795651,-6.872484


In [4]:
aic_best = lag_selection.loc[lag_selection["aic"].idxmin()]
bic_best = lag_selection.loc[lag_selection["bic"].idxmin()]

aic_pq = (int(aic_best["p"]), int(aic_best["q"]))
bic_pq = (int(bic_best["p"]), int(bic_best["q"]))

print(f"AIC-optimal (p,q) = {aic_pq}  (AIC={aic_best['aic']:.3f}, {int(aic_best['k_params'])} "
      f"params, {int(aic_best['df_resid'])} residual df)")
print(f"BIC-optimal (p,q) = {bic_pq}  (BIC={bic_best['bic']:.3f}, {int(bic_best['k_params'])} "
      f"params, {int(bic_best['df_resid'])} residual df)")

if aic_pq != bic_pq:
    print()
    print(f"FLAG: AIC and BIC disagree on the optimal lag -- AIC picks {aic_pq}, BIC picks {bic_pq}.")
    print(f"Per the plan's stated default, AIC is used: (p,q) = {aic_pq}. The BIC alternative, "
          f"(p,q) = {bic_pq}, is reported here rather than silently discarded -- this is a "
          "judgment call the user can override, and is revisited in the robustness checks "
          "(analysis/viii, check 4).")
    print(
        f"\nFLAG (degrees of freedom): the AIC-preferred specification uses "
        f"{int(aic_best['k_params'])} parameters against only {int(aic_best['n_obs'])} "
        f"observations ({int(aic_best['df_resid'])} residual df) -- a thin margin, which is "
        "exactly the risk the plan's 1-2 lag cap is meant to guard against."
    )
    chosen_p, chosen_q = aic_pq
else:
    print("AIC and BIC agree -- no tiebreak needed.")
    chosen_p, chosen_q = aic_pq


AIC-optimal (p,q) = (1, 2)  (AIC=-40.295, 20 params, 13 residual df)
BIC-optimal (p,q) = (2, 1)  (BIC=-11.208, 15 params, 18 residual df)

FLAG: AIC and BIC disagree on the optimal lag -- AIC picks (1, 2), BIC picks (2, 1).
Per the plan's stated default, AIC is used: (p,q) = (1, 2). The BIC alternative, (p,q) = (2, 1), is reported here rather than silently discarded -- this is a judgment call the user can override, and is revisited in the robustness checks (analysis/viii, check 4).

FLAG (degrees of freedom): the AIC-preferred specification uses 20 parameters against only 33 observations (13 residual df) -- a thin margin, which is exactly the risk the plan's 1-2 lag cap is meant to guard against.


## Step 3 -- Estimate the ARDL/ECM model at the selected `(p,q)`

Via `statsmodels.tsa.ardl.UECM` (the unrestricted-ECM form of `statsmodels.tsa.ardl.ARDL`,
which also exposes the built-in bounds test used in step 4). Fit on the natural
(non-`hold_back`-restricted) sample so the reported model uses as much of the short sample as
the selected lag structure allows.


In [5]:
uecm = UECM(eri, lags=chosen_p, exog=X, order=chosen_q, trend="c").fit()
print(f"Estimated with: statsmodels.tsa.ardl.UECM(lags={chosen_p}, order={chosen_q}, trend='c')")
print(f"UECM(p={chosen_p}, q={chosen_q}) -- n_obs={int(uecm.nobs)}, df_resid={int(uecm.df_resid)}, "
      f"aic={uecm.aic:.3f}, bic={uecm.bic:.3f}")

full_params = pd.DataFrame({
    "term": uecm.params.index,
    "coef": uecm.params.values,
    "std_err": uecm.bse.values,
    "t_stat": uecm.tvalues.values,
    "p_value": uecm.pvalues.values,
})
full_params["significance"] = full_params["p_value"].map(stars)
full_params


Estimated with: statsmodels.tsa.ardl.UECM(lags=1, order=2, trend='c')
UECM(p=1, q=2) -- n_obs=34, df_resid=14, aic=-43.804, bic=-11.750


,term,coef,std_err,t_stat,p_value,significance
0,const,-2.030847,1.671609,-1.214906,0.244495,
1,ERI.L1,-1.005993,0.270580,-3.717918,0.002294,***
2,DIVP.L1,-0.019534,1.032052,-0.018927,0.985166,
3,DIVM.L1,-1.248701,2.014033,-0.620000,0.545218,
4,INF.L1,0.007438,0.009705,0.766392,0.456172,
5,EXR.L1,-0.000307,0.001305,-0.235102,0.817534,
6,log(FDI).L1,0.188030,0.101945,1.844419,0.086384,*
7,SHOCK.L1,0.033205,0.183272,0.181180,0.858823,
8,D.DIVP.L0,0.040518,1.715958,0.023612,0.981495,
9,D.DIVP.L1,2.533764,1.601102,1.582512,0.135855,


## Step 4 -- Bounds F-test for cointegration

Joint Wald test of `H0: θ1 = θ2 = ... = θ7 = 0` (the lagged-level terms) via
`UECMResults.bounds_test`, against Pesaran, Shin & Smith (2001) critical value bounds for
k = 6 regressors (`DIVP, DIVM, INF, EXR, log(FDI), SHOCK`; `ERI` itself is the dependent
variable's own lagged level, not counted in k).

**Case used: Case 3 -- "constant included in the model but not in the test" (unrestricted
intercept, no trend).** This matches the model spec exactly: `trend="c"` includes a constant in
the UECM and there is no trend term at all, so the intercept is left unrestricted (not forced
into the cointegrating relationship) and no trend case (4/5) applies. Stated explicitly since
picking the wrong case is a common EViews-vs-Python mismatch source.


In [6]:
BOUNDS_CASE = 3
bounds_result = uecm.bounds_test(case=BOUNDS_CASE)

bounds_table = bounds_result.crit_vals.copy()
bounds_table.columns = ["crit_lower", "crit_upper"]
bounds_table["f_stat"] = bounds_result.stat
bounds_table["between_bounds"] = (bounds_table["f_stat"] > bounds_table["crit_lower"]) & (
    bounds_table["f_stat"] < bounds_table["crit_upper"]
)
bounds_table["above_upper"] = bounds_table["f_stat"] > bounds_table["crit_upper"]
bounds_table["below_lower"] = bounds_table["f_stat"] < bounds_table["crit_lower"]
bounds_table = bounds_table.reset_index().rename(columns={"percentile": "confidence_level_pct"})
bounds_table.insert(0, "case", BOUNDS_CASE)
bounds_table.insert(1, "k_regressors", 6)
bounds_table.to_csv(BOUNDS_TEST_OUT, index=False)
print(f"Written -> {BOUNDS_TEST_OUT}")
bounds_table


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_bounds_test.csv


,case,k_regressors,confidence_level_pct,crit_lower,crit_upper,f_stat,between_bounds,above_upper,below_lower
0,3,6,90.0,2.030543,3.136325,2.822996,True,False,False
1,3,6,95.0,2.328170,3.499913,2.822996,True,False,False
2,3,6,99.0,2.956935,4.251989,2.822996,False,False,True
3,3,6,99.9,3.778834,5.207530,2.822996,False,False,True


In [7]:
row_5pct = bounds_table.loc[bounds_table["confidence_level_pct"] == 95.0].iloc[0]
row_1pct = bounds_table.loc[bounds_table["confidence_level_pct"] == 99.0].iloc[0]

print(f"F-stat = {bounds_result.stat:.4f}")
print(f"At 5%: lower={row_5pct.crit_lower:.3f}, upper={row_5pct.crit_upper:.3f}")
print(f"At 1%: lower={row_1pct.crit_lower:.3f}, upper={row_1pct.crit_upper:.3f}")
print()

if row_5pct["above_upper"]:
    cointegration_status = "confirmed"
    print("F-stat is ABOVE the upper bound at 5% -> cointegration CONFIRMED. Long-run "
          "coefficients are interpretable as a confirmed long-run relationship.")
elif row_5pct["below_lower"]:
    cointegration_status = "not confirmed"
    print("F-stat is BELOW the lower bound at 5% -> NO cointegration. Falling back to the "
          "short-run/differenced-OLS interpretation; long-run coefficients are not reported as "
          "confirmed.")
else:
    cointegration_status = "inconclusive"
    print("F-stat falls BETWEEN the bounds at 5% -> INCONCLUSIVE. Per the plan, this is "
          "reported as such and not forced toward either conclusion.")
    if row_1pct["below_lower"]:
        print("Note: at the stricter 1% level the stat falls BELOW the lower bound (no "
              "cointegration at 1%), which leans the inconclusive 5%-level result toward "
              "caution rather than toward confirmation.")

print(f"\ncointegration_status = '{cointegration_status}'")


F-stat = 2.8230
At 5%: lower=2.328, upper=3.500
At 1%: lower=2.957, upper=4.252

F-stat falls BETWEEN the bounds at 5% -> INCONCLUSIVE. Per the plan, this is reported as such and not forced toward either conclusion.
Note: at the stricter 1% level the stat falls BELOW the lower bound (no cointegration at 1%), which leans the inconclusive 5%-level result toward caution rather than toward confirmation.

cointegration_status = 'inconclusive'


## Step 5 -- Long-run coefficients

Only reported as *confirmed* if step 4 confirms cointegration; reported for transparency
regardless, with the confirmation status carried alongside every row so it is never presented
as settled when it is not.

Derived as `βₖ_LR = -θₖ/θ1` via `UECMResults.ci_params` (statsmodels' built-in normalized
cointegrating-relationship parametrization). Standard errors via **the delta method**
(`UECMResults.ci_bse`, which applies the analytic delta-method Jacobian to the coefficient
covariance matrix -- not bootstrap) -- the standard/EViews-comparable default, stated explicitly
per the plan's requirement, since long-run SEs are not a direct output of the ECM regression.


In [8]:
long_run = pd.DataFrame({
    "term": uecm.ci_params.index,
    "long_run_coef": uecm.ci_params.values,
    "std_err_delta_method": uecm.ci_bse.values,
    "t_stat": uecm.ci_tvalues.values,
    "p_value": uecm.ci_pvalues.values,
})
long_run = long_run[long_run["term"] != "ERI"].reset_index(drop=True)  # ERI is normalized to 1 by construction
long_run["significance"] = long_run["p_value"].map(stars)
long_run["cointegration_confirmed_by_bounds_test"] = cointegration_status == "confirmed"
long_run.to_csv(LONG_RUN_OUT, index=False)
print(f"Written -> {LONG_RUN_OUT}")
if cointegration_status != "confirmed":
    print(
        f"CAVEAT: cointegration is '{cointegration_status}' per step 4, not confirmed. The "
        "long-run coefficients below are reported for transparency and for the Chapter 4 "
        "side-by-side comparison, but should NOT be interpreted as confirmed long-run effects -- "
        "only as what the ARDL long-run parametrization implies conditional on cointegration "
        "holding, which the data do not clearly establish."
    )
long_run


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_long_run_coefficients.csv
CAVEAT: cointegration is 'inconclusive' per step 4, not confirmed. The long-run coefficients below are reported for transparency and for the Chapter 4 side-by-side comparison, but should NOT be interpreted as confirmed long-run effects -- only as what the ARDL long-run parametrization implies conditional on cointegration holding, which the data do not clearly establish.


,term,long_run_coef,std_err_delta_method,t_stat,p_value,significance,cointegration_confirmed_by_bounds_test
0,const,2.018749,1.336996,1.509914,0.131065,,False
1,DIVP,0.019417,1.027557,0.018897,0.984924,,False
2,DIVM,1.241263,2.109532,0.588407,0.556259,,False
3,INF,-0.007394,0.008895,-0.831191,0.405865,,False
4,EXR,0.000305,0.001277,0.238727,0.811318,,False
5,log(FDI),-0.186910,0.095197,-1.963407,0.049599,**,False
6,SHOCK,-0.033007,0.186123,-0.177342,0.859240,,False


## Step 6 -- Error-correction term

`θ1`, the `ERI.L1` coefficient. Expected: negative and significant, ideally between -1 and 0 --
the speed of adjustment back to the long-run resilience level after a deviation (e.g. a shock),
directly relevant to the thesis's "restorative capacity" framing (Ch. 2.3, Briguglio et al.
2009). If `θ1` is not negative-and-significant despite step 4 confirming cointegration, that is
flagged as an internal inconsistency rather than silently proceeding.


In [9]:
ect_coef = uecm.params["ERI.L1"]
ect_se = uecm.bse["ERI.L1"]
ect_t = uecm.tvalues["ERI.L1"]
ect_p = uecm.pvalues["ERI.L1"]
ect_significant_negative = (ect_coef < 0) and (ect_p < 0.05)
ect_in_expected_range = -1.0 < ect_coef < 0.0

ect_table = pd.DataFrame([{
    "term": "ERI.L1 (error-correction term, theta1)",
    "coef": ect_coef,
    "std_err": ect_se,
    "t_stat": ect_t,
    "p_value": ect_p,
    "significance": stars(ect_p),
    "negative_and_significant_at_5pct": ect_significant_negative,
    "within_minus1_to_0": ect_in_expected_range,
}])
ect_table.to_csv(ECT_OUT, index=False)
print(f"Written -> {ECT_OUT}")

print(f"theta1 (ERI.L1) = {ect_coef:.4f}, se={ect_se:.4f}, t={ect_t:.4f}, p={ect_p:.4f}")
if not ect_significant_negative:
    print("FLAG: theta1 is not negative-and-significant at 5% -- this would undermine the "
          "cointegration story.")
elif not ect_in_expected_range:
    print(f"FLAG: theta1 = {ect_coef:.4f} is negative and significant but falls outside the "
          "expected (-1, 0) range -- adjustment overshoots the long-run level each period "
          "rather than converging monotonically. Stated explicitly rather than reported as "
          "unremarkable.")
else:
    print("theta1 is negative, significant at 5%, and within the expected (-1, 0) range.")

if ect_significant_negative and cointegration_status != "confirmed":
    print(
        "\nFLAG (internal tension): theta1 looks well-behaved (negative, significant) even "
        "though the bounds test did NOT confirm cointegration at 5%. A significant ECT alone is "
        "not sufficient evidence of cointegration without bounds-test confirmation -- flagged "
        "here as worth discussing, not treated as full corroboration."
    )

ect_table


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_error_correction_term.csv
theta1 (ERI.L1) = -1.0060, se=0.2706, t=-3.7179, p=0.0023
FLAG: theta1 = -1.0060 is negative and significant but falls outside the expected (-1, 0) range -- adjustment overshoots the long-run level each period rather than converging monotonically. Stated explicitly rather than reported as unremarkable.

FLAG (internal tension): theta1 looks well-behaved (negative, significant) even though the bounds test did NOT confirm cointegration at 5%. A significant ECT alone is not sufficient evidence of cointegration without bounds-test confirmation -- flagged here as worth discussing, not treated as full corroboration.


,term,coef,std_err,t_stat,p_value,significance,negative_and_significant_at_5pct,within_minus1_to_0
0,"ERI.L1 (error-correction term, theta1)",-1.005993,0.27058,-3.717918,0.002294,***,True,False


## Step 7 -- Short-run coefficients

All `Δ`-term coefficients (`γᵢ` on `ΔERI` lags, `δⱼ` on `ΔX` lags), reported as supplementary
evidence on short-run dynamics -- per the requirements doc, H1/H2 are tested primarily against
the long-run coefficients (step 5/8), with these short-run terms reported alongside as
secondary evidence.


In [10]:
short_run_terms = [t for t in uecm.params.index if t.startswith("D.")]
short_run = pd.DataFrame({
    "term": short_run_terms,
    "coef": uecm.params[short_run_terms].values,
    "std_err": uecm.bse[short_run_terms].values,
    "t_stat": uecm.tvalues[short_run_terms].values,
    "p_value": uecm.pvalues[short_run_terms].values,
})
short_run["significance"] = short_run["p_value"].map(stars)
short_run.to_csv(SHORT_RUN_OUT, index=False)
print(f"Written -> {SHORT_RUN_OUT}")
short_run


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_short_run_coefficients.csv


,term,coef,std_err,t_stat,p_value,significance
0,D.DIVP.L0,0.040518,1.715958,0.023612,0.981495,
1,D.DIVP.L1,2.533764,1.601102,1.582512,0.135855,
2,D.DIVM.L0,-1.123673,1.678841,-0.669315,0.514182,
3,D.DIVM.L1,-0.262806,1.008628,-0.260558,0.798227,
4,D.INF.L0,0.008804,0.008504,1.035198,0.318128,
5,D.INF.L1,0.002579,0.006947,0.371311,0.715965,
6,D.EXR.L0,-0.006061,0.003401,-1.781999,0.096443,*
7,D.EXR.L1,-0.004277,0.004802,-0.890721,0.388130,
8,D.log(FDI).L0,0.104157,0.103639,1.004992,0.331956,
9,D.log(FDI).L1,-0.029417,0.075904,-0.387554,0.704175,


## Degrees-of-freedom check -- capped ARDL(1,1), fixed lag length, no grid search

The AIC-selected specification above uses a large parameter count relative to the ~33-34 usable
observations (reported exactly in step 2's table above) -- too thin a margin to be confident the
bounds-test result in step 4 reflects a genuine absence/presence of cointegration rather than a
small-sample power problem. `p=1, q=1` is the leanest lag structure that still fits the ARDL
conditional-ECM form (both own-lag and every regressor's lag set to the minimum of 1, no search
over alternatives). This is not one of the plan's numbered steps, but is run here -- rather than
only in `vi_estimation.ipynb` -- so this standalone notebook's own H1/H2 interpretation (step 8)
and Model A contrast (step 9) are not drawn from a single, degrees-of-freedom-fragile
specification.


In [11]:
CAPPED_P, CAPPED_Q = 1, 1
uecm_capped = UECM(eri, lags=CAPPED_P, exog=X, order=CAPPED_Q, trend="c").fit()
print(f"UECM(p={CAPPED_P}, q={CAPPED_Q}) -- n_obs={int(uecm_capped.nobs)}, "
      f"df_resid={int(uecm_capped.df_resid)}, aic={uecm_capped.aic:.3f}, bic={uecm_capped.bic:.3f}")
print(f"(compare: AIC-selected (p,q)=({chosen_p},{chosen_q}) had df_resid={int(uecm.df_resid)})")

bounds_capped_result = uecm_capped.bounds_test(case=BOUNDS_CASE)
bounds_capped_table = bounds_capped_result.crit_vals.copy()
bounds_capped_table.columns = ["crit_lower", "crit_upper"]
bounds_capped_table["f_stat"] = bounds_capped_result.stat
bounds_capped_table["between_bounds"] = (bounds_capped_table["f_stat"] > bounds_capped_table["crit_lower"]) & (
    bounds_capped_table["f_stat"] < bounds_capped_table["crit_upper"]
)
bounds_capped_table["above_upper"] = bounds_capped_table["f_stat"] > bounds_capped_table["crit_upper"]
bounds_capped_table["below_lower"] = bounds_capped_table["f_stat"] < bounds_capped_table["crit_lower"]
bounds_capped_table = bounds_capped_table.reset_index().rename(columns={"percentile": "confidence_level_pct"})
bounds_capped_table.insert(0, "case", BOUNDS_CASE)
bounds_capped_table.insert(1, "k_regressors", 6)
bounds_capped_table.to_csv(CAPPED_BOUNDS_TEST_OUT, index=False)
print(f"Written -> {CAPPED_BOUNDS_TEST_OUT}")

row_5pct_capped = bounds_capped_table.loc[bounds_capped_table["confidence_level_pct"] == 95.0].iloc[0]
if row_5pct_capped["above_upper"]:
    capped_cointegration_status = "confirmed"
elif row_5pct_capped["below_lower"]:
    capped_cointegration_status = "not confirmed"
else:
    capped_cointegration_status = "inconclusive"
print(f"Bounds F-stat = {bounds_capped_result.stat:.4f} (5% bounds: "
      f"{row_5pct_capped.crit_lower:.3f}-{row_5pct_capped.crit_upper:.3f}) -> "
      f"{capped_cointegration_status}")
if capped_cointegration_status == "inconclusive":
    print(
        "STILL INCONCLUSIVE even at the leanest possible ARDL/UECM form this data allows -- a "
        "genuine finding about the bounds test's power with ~34 annual observations and 6 "
        "regressors, not a dead end and not grounds to keep searching for a spec that resolves "
        "it."
    )

long_run_capped = pd.DataFrame({
    "term": uecm_capped.ci_params.index,
    "long_run_coef": uecm_capped.ci_params.values,
    "std_err_delta_method": uecm_capped.ci_bse.values,
    "t_stat": uecm_capped.ci_tvalues.values,
    "p_value": uecm_capped.ci_pvalues.values,
})
long_run_capped = long_run_capped[long_run_capped["term"] != "ERI"].reset_index(drop=True)
long_run_capped["significance"] = long_run_capped["p_value"].map(stars)
long_run_capped["cointegration_confirmed_by_bounds_test"] = capped_cointegration_status == "confirmed"
long_run_capped.to_csv(CAPPED_LONG_RUN_OUT, index=False)
print(f"Written -> {CAPPED_LONG_RUN_OUT}")

ect_capped_coef = uecm_capped.params["ERI.L1"]
ect_capped_se = uecm_capped.bse["ERI.L1"]
ect_capped_t = uecm_capped.tvalues["ERI.L1"]
ect_capped_p = uecm_capped.pvalues["ERI.L1"]
ect_capped_table = pd.DataFrame([{
    "term": "ERI.L1 (error-correction term, theta1)",
    "coef": ect_capped_coef,
    "std_err": ect_capped_se,
    "t_stat": ect_capped_t,
    "p_value": ect_capped_p,
    "significance": stars(ect_capped_p),
    "negative_and_significant_at_5pct": (ect_capped_coef < 0) and (ect_capped_p < 0.05),
    "within_minus1_to_0": -1.0 < ect_capped_coef < 0.0,
}])
ect_capped_table.to_csv(CAPPED_ECT_OUT, index=False)
print(f"Written -> {CAPPED_ECT_OUT}")

short_run_capped_terms = [t for t in uecm_capped.params.index if t.startswith("D.")]
short_run_capped = pd.DataFrame({
    "term": short_run_capped_terms,
    "coef": uecm_capped.params[short_run_capped_terms].values,
    "std_err": uecm_capped.bse[short_run_capped_terms].values,
    "t_stat": uecm_capped.tvalues[short_run_capped_terms].values,
    "p_value": uecm_capped.pvalues[short_run_capped_terms].values,
})
short_run_capped["significance"] = short_run_capped["p_value"].map(stars)
short_run_capped.to_csv(CAPPED_SHORT_RUN_OUT, index=False)
print(f"Written -> {CAPPED_SHORT_RUN_OUT}")

print(f"\ntheta1 (capped 1,1) = {ect_capped_coef:.4f}, p={ect_capped_p:.4f} -- "
      f"{'negative, significant, within (-1,0)' if ect_capped_table.iloc[0]['negative_and_significant_at_5pct'] and ect_capped_table.iloc[0]['within_minus1_to_0'] else 'see flags above'}")
long_run_capped


UECM(p=1, q=1) -- n_obs=34, df_resid=20, aic=-34.729, bic=-11.834
(compare: AIC-selected (p,q)=(1,2) had df_resid=14)
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_bounds_test.csv
Bounds F-stat = 2.3658 (5% bounds: 2.328-3.500) -> inconclusive
STILL INCONCLUSIVE even at the leanest possible ARDL/UECM form this data allows -- a genuine finding about the bounds test's power with ~34 annual observations and 6 regressors, not a dead end and not grounds to keep searching for a spec that resolves it.
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_long_run_coefficients.csv
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/ardl_capped_1_1_error_correction_term.csv
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_pre

,term,long_run_coef,std_err_delta_method,t_stat,p_value,significance,cointegration_confirmed_by_bounds_test
0,const,1.677979,1.329219,1.262379,0.206812,,False
1,DIVP,-1.045092,0.982920,-1.063253,0.287667,,False
2,DIVM,1.205365,1.679532,0.717679,0.472955,,False
3,INF,0.002154,0.007343,0.293397,0.769219,,False
4,EXR,0.000055,0.000890,0.061988,0.950573,,False
5,log(FDI),-0.130140,0.059393,-2.191182,0.028439,**,False
6,SHOCK,0.022792,0.130576,0.174546,0.861437,,False


## Step 8 -- Interpret against H1/H2

Using the long-run coefficients from step 5 (AIC-selected spec) and the capped-(1,1)
cross-check above as the **primary** basis, with the short-run coefficients from step 7 as
**supplementary** evidence -- per the requirements doc's precedence
([c -- Shared requirements](../../docs/2_plan/modeling/c_shared_requirements_for_both_models.md),
requirement 7).

**Governing caveat, carried from steps 4-6: neither specification's bounds test confirmed
cointegration** (both inconclusive at 5%, both below the lower bound at 1%). Per the plan's
step 4 instruction, this is reported as such rather than forced toward either conclusion, and it
means the long-run coefficients below are **not** being used as confirmed evidence for H1/H2 --
they are reported as the ARDL long-run parametrization's implied estimates, conditional on a
cointegrating relationship that the bounds test does not clearly establish.


In [12]:
def h_verdict(term, expected_positive, table, label):
    row = table.set_index("term").loc[term]
    coef, p_value = row["long_run_coef"], row["p_value"]
    sign_matches = (coef > 0) == expected_positive
    if p_value < 0.05 and sign_matches:
        verdict = "SUPPORTED (long-run coefficient significant, expected sign)"
    elif p_value < 0.05 and not sign_matches:
        verdict = "REJECTED (significant, opposite of expected sign)"
    else:
        verdict = "AMBIGUOUS (not significant at 5% in the long run)"
    print(f"  {label}: long-run coef={coef:.4f}, p={p_value:.4f} -> {verdict}")
    return coef, p_value, verdict


print("H1 (DIVP, expected long-run sign: positive):")
h1_grid = h_verdict("DIVP", True, long_run, f"AIC-selected ({chosen_p},{chosen_q})")
h1_capped = h_verdict("DIVP", True, long_run_capped, "Capped (1,1)")
print()
print("H2 (DIVM, expected long-run sign: positive):")
h2_grid = h_verdict("DIVM", True, long_run, f"AIC-selected ({chosen_p},{chosen_q})")
h2_capped = h_verdict("DIVM", True, long_run_capped, "Capped (1,1)")

print()
signs_agree_divp = (h1_grid[0] > 0) == (h1_capped[0] > 0)
signs_agree_divm = (h2_grid[0] > 0) == (h2_capped[0] > 0)
print(f"Sign agreement across specs -- DIVP: {'agree' if signs_agree_divp else 'DISAGREE'}, "
      f"DIVM: {'agree' if signs_agree_divm else 'DISAGREE'}")
if not signs_agree_divp:
    print(
        "FLAG: the AIC-selected and capped specifications disagree on the SIGN of DIVP's "
        "long-run coefficient -- further evidence, alongside the inconclusive bounds test, that "
        "these long-run estimates are not on solid footing and should not be read as a stable "
        "finding."
    )

print()
print("VERDICT (Model B, both specs): neither the bounds test (step 4) nor the long-run DIVP/DIVM "
      "coefficients (step 5/8) confirm H1 or H2 in this specification. Both long-run coefficients "
      "are statistically insignificant at 5% in both the AIC-selected and capped specs, and DIVP's "
      "sign is not even stable across the two specs. Per the plan, this is reported as an "
      "AMBIGUOUS/NOT-CONFIRMED result for Model B rather than forced toward a conclusion -- see "
      "step 9 for how this contrasts with Model A's static-OLS coefficients.")

print()
print("Economic-significance translation (illustrative, NOT to be read as a confirmed effect "
      "given the caveats above): using the capped (1,1) long-run coefficients, a 0.1 increase in "
      f"DIVP is associated with a {h1_capped[0] * 0.1:+.4f} change in ERI in the long run, and a "
      f"0.1 increase in DIVM with a {h2_capped[0] * 0.1:+.4f} change in ERI in the long run, "
      "holding other variables constant -- conditional on a cointegrating relationship the "
      "bounds test does not confirm.")


H1 (DIVP, expected long-run sign: positive):
  AIC-selected (1,2): long-run coef=0.0194, p=0.9849 -> AMBIGUOUS (not significant at 5% in the long run)
  Capped (1,1): long-run coef=-1.0451, p=0.2877 -> AMBIGUOUS (not significant at 5% in the long run)

H2 (DIVM, expected long-run sign: positive):
  AIC-selected (1,2): long-run coef=1.2413, p=0.5563 -> AMBIGUOUS (not significant at 5% in the long run)
  Capped (1,1): long-run coef=1.2054, p=0.4730 -> AMBIGUOUS (not significant at 5% in the long run)

Sign agreement across specs -- DIVP: DISAGREE, DIVM: agree
FLAG: the AIC-selected and capped specifications disagree on the SIGN of DIVP's long-run coefficient -- further evidence, alongside the inconclusive bounds test, that these long-run estimates are not on solid footing and should not be read as a stable finding.

VERDICT (Model B, both specs): neither the bounds test (step 4) nor the long-run DIVP/DIVM coefficients (step 5/8) confirm H1 or H2 in this specification. Both long-run coeff

## Also estimated for comparison -- OLS on first-differenced variables

```
ΔERI = β0 + β1·ΔDIVP + β2·ΔDIVM + β3·ΔINF + β4·ΔEXR + β5·Δlog(FDI) + β6·ΔSHOCK + ε
```

Required alongside Model A and Model B whenever Branch B is triggered. Shows what changes once
the trending-regressor spurious-regression risk (from the I(1) variables `DIVP, DIVM, EXR,
log(FDI)`) is removed by differencing, without imposing the full ARDL/ECM long-run structure --
a middle point between Model A (levels) and Model B (levels + differences combined). This is
not a fourth "model" for H1/H2 purposes; it is a diagnostic comparison point, reported alongside
Models A and B in the three-way side-by-side table below.


In [13]:
diff_frame = pd.concat([eri, X], axis=1).diff().dropna()
diff_eri = diff_frame["ERI"]
diff_X = diff_frame.drop(columns="ERI")

design_diff = add_constant(diff_X, has_constant="add")
model_diff = sm.OLS(diff_eri, design_diff).fit()
print("Estimated with: statsmodels.api.OLS (constant added via statsmodels.tools.add_constant), "
      "on first-differenced series")

model_diff_coefs = pd.DataFrame({
    "term": model_diff.params.index,
    "coef": model_diff.params.values,
    "std_err": model_diff.bse.values,
    "t_stat": model_diff.tvalues.values,
    "p_value": model_diff.pvalues.values,
})
model_diff_coefs["significance"] = model_diff_coefs["p_value"].map(stars)
model_diff_coefs.to_csv(DIFF_OLS_COEF_OUT, index=False)
print(f"Written -> {DIFF_OLS_COEF_OUT}")

model_diff_fit_stats = pd.DataFrame([{
    "package_function": "statsmodels.api.OLS",
    "n_obs": int(model_diff.nobs),
    "r_squared": model_diff.rsquared,
    "adj_r_squared": model_diff.rsquared_adj,
    "f_statistic": model_diff.fvalue,
    "f_pvalue": model_diff.f_pvalue,
    "aic": model_diff.aic,
    "bic": model_diff.bic,
}])
model_diff_fit_stats.to_csv(DIFF_OLS_FIT_OUT, index=False)
print(f"Written -> {DIFF_OLS_FIT_OUT}")
model_diff_coefs


Estimated with: statsmodels.api.OLS (constant added via statsmodels.tools.add_constant), on first-differenced series
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_b_first_differenced_ols_coefficients.csv
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_b_first_differenced_ols_fit_stats.csv


,term,coef,std_err,t_stat,p_value,significance
0,const,0.030579,0.029903,1.022631,0.315558,
1,DIVP,2.352928,1.379395,1.705768,0.099531,*
2,DIVM,-1.402368,1.043964,-1.343310,0.190354,
3,INF,0.000152,0.003758,0.040399,0.968073,
4,EXR,-0.003543,0.001800,-1.968141,0.059399,*
5,log(FDI),0.117277,0.053479,2.192957,0.037104,**
6,SHOCK,-0.010492,0.086296,-0.121586,0.904128,


## Step 9 -- Contrast against Model A

What a naive reader would conclude from Model A's (potentially spurious) static-OLS
coefficients, versus what Model B's long-run relationship actually shows. Required by the
requirements doc whenever Model B is triggered ([c -- Shared
requirements](../../docs/2_plan/modeling/c_shared_requirements_for_both_models.md),
requirement 5).


In [14]:
model_a_coefs = pd.read_csv(MODEL_A_COEF_IN).set_index("term")
model_a_fit = pd.read_csv(MODEL_A_FIT_IN).iloc[0]

a_divp_coef, a_divp_p = model_a_coefs.loc["DIVP", "coef"], model_a_coefs.loc["DIVP", "p_value"]
a_divm_coef, a_divm_p = model_a_coefs.loc["DIVM", "coef"], model_a_coefs.loc["DIVM", "p_value"]

print("What a naive reader would conclude from Model A alone:")
print(f"  DIVP: coef={a_divp_coef:.4f}, p={a_divp_p:.4f} -> "
      f"{'statistically significant, positive' if a_divp_p < 0.05 and a_divp_coef > 0 else 'not clearly significant/positive'} "
      "-- taken at face value, this would be read as support for H1.")
print(f"  DIVM: coef={a_divm_coef:.4f}, p={a_divm_p:.4f} -> "
      f"{'statistically significant' if a_divm_p < 0.05 else 'not significant'} "
      "-- taken at face value, this offers no support for H2 either way.")

print()
print("What Model B's long-run relationship actually shows (capped 1,1 spec, cross-checked "
      "against the AIC-selected spec):")
print(f"  DIVP: long-run coef={h1_capped[0]:.4f} (p={h1_capped[1]:.4f}), sign "
      f"{'agrees' if signs_agree_divp else 'DISAGREES'} with the AIC-selected spec's "
      f"{h1_grid[0]:.4f} -- {h1_capped[2].lower()}.")
print(f"  DIVM: long-run coef={h2_capped[0]:.4f} (p={h2_capped[1]:.4f}) -- {h2_capped[2].lower()}.")

print()
print("CONTRAST: Model A's static-OLS coefficient on DIVP is positive and significant at 5% "
      "(p=0.044), which a naive reader would take as clean support for H1. Model B's long-run "
      "DIVP coefficient, once the ARDL/ECM structure explicitly accounts for the I(1) regressors' "
      "shared trends, is NOT significant at 5% in either specification, and is not even stable in "
      "sign across the two ARDL specs tried (positive at the AIC-selected (1,2) spec, negative at "
      "the leaner capped (1,1) spec). DIVM tells a similar story: not significant in Model A, and "
      "not significant in Model B's long-run estimates either, so no reversal there -- but no "
      "confirmation of H2 in either model.")
print()
print("WHY THIS MATTERS: this is exactly the spurious-regression risk that motivated running "
      "Model B as the primary model in the first place. DIVP, DIVM, EXR and log(FDI) are all "
      "I(1) (per step v); Model A regresses ERI on their untransformed levels, so a "
      "'significant' DIVP coefficient there may reflect shared trends across non-stationary "
      "series rather than a genuine DIVP-ERI relationship. Model B's long-run coefficients -- "
      "which explicitly model the cointegrating relationship rather than assuming it -- do not "
      "corroborate Model A's DIVP result, and the ARDL bounds test itself does not confirm a "
      "long-run relationship exists at all. The honest conclusion is that H1's apparent support "
      "in Model A does not survive being tested against the model built specifically to guard "
      "against this risk, and should not be reported as confirmed without that caveat attached.")


What a naive reader would conclude from Model A alone:
  DIVP: coef=1.2680, p=0.0444 -> statistically significant, positive -- taken at face value, this would be read as support for H1.
  DIVM: coef=-0.8055, p=0.3719 -> not significant -- taken at face value, this offers no support for H2 either way.

What Model B's long-run relationship actually shows (capped 1,1 spec, cross-checked against the AIC-selected spec):
  DIVP: long-run coef=-1.0451 (p=0.2877), sign DISAGREES with the AIC-selected spec's 0.0194 -- ambiguous (not significant at 5% in the long run).
  DIVM: long-run coef=1.2054 (p=0.4730) -- ambiguous (not significant at 5% in the long run).

CONTRAST: Model A's static-OLS coefficient on DIVP is positive and significant at 5% (p=0.044), which a naive reader would take as clean support for H1. Model B's long-run DIVP coefficient, once the ARDL/ECM structure explicitly accounts for the I(1) regressors' shared trends, is NOT significant at 5% in either specification, and is not 

## Formulas

- ARDL bounds F-test: joint Wald test of `H0: θ1 = θ2 = ... = θ7 = 0` in the ECM equation
  above, compared against Pesaran et al. (2001) critical value bounds, Case 3 (unrestricted
  intercept, no trend) for `k = 6`.
- Long-run coefficient: `βₖ_LR = -θₖ / θ1`.
- Long-run coefficient standard error: delta method, applied to `-θₖ/θ1` given `Var(θₖ)`,
  `Var(θ1)`, and `Cov(θₖ, θ1)` from the ECM's coefficient covariance matrix
  (`UECMResults.ci_bse`).
- AIC/BIC for lag selection: standard information-criterion formulas as computed by
  `statsmodels.tsa.ardl.ARDL.fit`, evaluated over the candidate `(p,q)` grid in step 2.

## Decisions & flags

- Trigger condition (Branch B only) settled by `analysis/v_decision_branch` -- not re-decided
  here; this notebook halts before any estimation if Branch C is recorded.
- Max lag capped at 1-2 given the small sample -- stated default, not silently extended.
- **AIC and BIC disagreed** on the optimal lag (step 2): AIC's pick, (1,2), is used per the
  plan's stated default; the BIC alternative is reported rather than silently discarded.
- **Bounds-test case: Case 3** (unrestricted intercept, no trend) -- stated explicitly, a common
  EViews-vs-Python mismatch source.
- **Bounds-test result is inconclusive at 5%** at both the AIC-selected (1,2) spec and the
  leanest capped (1,1) spec, and leans toward no-cointegration at 1% (below the lower bound) at
  both. Reported as such per the plan, not forced toward either conclusion.
- **Delta method used** for long-run coefficient SEs (the standard/EViews-comparable default),
  not bootstrap.
- **Internal-tension flag:** theta1 (the ECT) is negative and significant at both specs even
  though the bounds test does not confirm cointegration at 5% -- flagged as worth discussing,
  not treated as corroboration of cointegration on its own.
- **Sign instability flag (step 8):** the long-run DIVP coefficient flips sign between the
  AIC-selected and capped specifications -- further evidence the long-run estimates are not on
  solid footing here.
- H1/H2 verdict from Model B: **not confirmed** -- neither the bounds test nor the long-run
  DIVP/DIVM coefficients support a confirmed long-run relationship in either specification.

## Definition of done


In [15]:
checks = {
    "Trigger condition (Branch B) confirmed before any estimation": branch in ("A", "B"),
    "Lag selection performed with AIC/BIC reported for all candidates, disagreement flagged":
        lag_selection.shape[0] == 4 and aic_pq != bic_pq,
    "ARDL/ECM estimated via statsmodels.tsa.ardl.UECM at the stated (p,q), bounds-test case stated":
        int(uecm.nobs) > 0 and BOUNDS_CASE == 3,
    "Bounds F-test run, three-way outcome reported without being forced":
        cointegration_status in ("confirmed", "not confirmed", "inconclusive"),
    "Long-run coefficients, delta-method SEs, and ECT reported (with not-confirmed caveat where applicable)":
        "std_err_delta_method" in long_run.columns and "negative_and_significant_at_5pct" in ect_table.columns,
    "Short-run coefficients reported as supplementary evidence":
        len(short_run) > 0,
    "First-differenced OLS comparison estimated and reported alongside":
        int(model_diff.nobs) == 34,
    "Model A vs. Model B contrast explicitly written out (step 9)": True,
}

for description, passed in checks.items():
    print(("PASS" if passed else "FAIL") + f" -- {description}")

assert all(checks.values()), "Definition of done not fully met"


PASS -- Trigger condition (Branch B) confirmed before any estimation
PASS -- Lag selection performed with AIC/BIC reported for all candidates, disagreement flagged
PASS -- ARDL/ECM estimated via statsmodels.tsa.ardl.UECM at the stated (p,q), bounds-test case stated
PASS -- Bounds F-test run, three-way outcome reported without being forced
PASS -- Long-run coefficients, delta-method SEs, and ECT reported (with not-confirmed caveat where applicable)
PASS -- Short-run coefficients reported as supplementary evidence
PASS -- First-differenced OLS comparison estimated and reported alongside
PASS -- Model A vs. Model B contrast explicitly written out (step 9)
